In [ ]:
%cd san-v2

In [ ]:
!pwd

In [ ]:
# Configuration parameters (equivalent to command-line arguments)
opts = dnnlib.EasyDict({
    'outdir': './runs/wc-cv_h200',
    'cfg': 'stylegan3-r',
    'data': './datasets/imagenet_9to4_1024x1024_16x16.zip',
    'gpus': 4,
    'batch': 1280,
    'mirror': False,  # 0 -> False
    'snap': 500,
    'batch_gpu': 320,
    'kimg': 20000,
    'cond': True,
    'syn_layers': 6,
    # Default values for other parameters
    'resume': None,
    'freezed': 0,
    'cbase': 32768,
    'cmax': 512,
    'glr': None,
    'dlr': 0.002,
    'map_depth': None,
    'desc': None,
    'metrics': 'fid50k_full',
    'tick': 4,
    'seed': 0,
    'fp32': False,
    'nobench': False,
    'workers': 3,
    'dry_run': False,
    'restart_every': 999999999,
    'stem': False,
    'superres': False,
    'path_stem': None,
    'head_layers': 7,
    'cls_weight': 0.0,
    'up_factor': 2,
})

# Initialize config.
c = dnnlib.EasyDict()  # Main config dict.
c.G_kwargs = dnnlib.EasyDict(class_name=None, z_dim=512, w_dim=512, mapping_kwargs=dnnlib.EasyDict())
c.G_opt_kwargs = dnnlib.EasyDict(class_name='torch.optim.Adam', betas=[0, 0.99], eps=1e-8)
c.D_opt_kwargs = dnnlib.EasyDict(class_name='torch.optim.Adam', betas=[0, 0.99], eps=1e-8)
c.data_loader_kwargs = dnnlib.EasyDict(pin_memory=True, prefetch_factor=2)

# Training set.
c.training_set_kwargs, dataset_name = init_dataset_kwargs(data=opts.data)
if opts.cond and not c.training_set_kwargs.use_labels:
    raise Exception('--cond=True requires labels specified in dataset.json')
c.training_set_kwargs.use_labels = opts.cond
c.training_set_kwargs.xflip = opts.mirror

# Hyperparameters & settings.
c.num_gpus = opts.gpus
c.batch_size = opts.batch
c.batch_gpu = opts.batch_gpu or opts.batch // opts.gpus
c.G_kwargs.channel_base = opts.cbase
c.G_kwargs.channel_max = opts.cmax
c.G_opt_kwargs.lr = (0.002 if opts.cfg == 'stylegan2' else 0.0025) if opts.glr is None else opts.glr
c.D_opt_kwargs.lr = opts.dlr
c.metrics = parse_comma_separated_list(opts.metrics)
c.total_kimg = opts.kimg
c.kimg_per_tick = opts.tick
c.image_snapshot_ticks = c.network_snapshot_ticks = opts.snap
c.random_seed = c.training_set_kwargs.random_seed = opts.seed
c.data_loader_kwargs.num_workers = opts.workers

# Sanity checks.
if c.batch_size % c.num_gpus != 0:
    raise Exception('--batch must be a multiple of --gpus')
if c.batch_size % (c.num_gpus * c.batch_gpu) != 0:
    raise Exception('--batch must be a multiple of --gpus times --batch-gpu')
if any(not metric_main.is_valid_metric(metric) for metric in c.metrics):
    raise Exception('\n'.join(['--metrics can only contain the following values:'] + metric_main.list_valid_metrics()))

# Base configuration.
c.ema_kimg = c.batch_size * 10 / 32
if opts.cfg == 'stylegan2':
    c.G_kwargs.class_name = 'training.networks_stylegan2.Generator'
    c.G_reg_interval = 4  # Enable lazy regularization for G.
    c.G_kwargs.fused_modconv_default = 'inference_only' # Speed up training by using regular convolutions instead of grouped convolutions.

elif opts.cfg == 'fastgan':
    c.G_kwargs = dnnlib.EasyDict(class_name='training.networks_fastgan.Generator',
                                 cond=opts.cond, mapping_kwargs=dnnlib.EasyDict(),
                                 synthesis_kwargs=dnnlib.EasyDict())
    c.G_kwargs.synthesis_kwargs.lite = True
    c.G_opt_kwargs.lr = c.D_opt_kwargs.lr = 0.0002
    c.G_opt_kwargs.lr = 0.002

else:
    c.G_kwargs.class_name = 'training.networks_stylegan3_resetting.Generator'
    c.G_kwargs.magnitude_ema_beta = 0.5 ** (c.batch_size / (20 * 1e3))
    c.G_kwargs.channel_base *= 2  # increase for StyleGAN-XL
    c.G_kwargs.channel_max *= 2   # increase for StyleGAN-XL
    c.G_kwargs.conv_kernel = 1 if opts.cfg == 'stylegan3-r' else 3
    c.G_kwargs.use_radial_filters = True if opts.cfg == 'stylegan3-r' else False

    if opts.cfg == 'stylegan3-r':
        c.G_kwargs.channel_base *= 2
        c.G_kwargs.channel_max *= 2

# Resume.
if opts.resume is not None:
    c.resume_pkl = opts.resume
    c.ada_kimg = 100  # Make ADA react faster at the beginning.
    c.ema_rampup = None  # Disable EMA rampup.

# Restart.
c.restart_every = opts.restart_every

# Performance-related toggles.
if opts.fp32:
    c.G_kwargs.num_fp16_res = 0
    c.G_kwargs.conv_clamp = None
if opts.nobench:
    c.cudnn_benchmark = False

# Description string.
desc = f'{opts.cfg:s}-{dataset_name:s}-gpus{c.num_gpus:d}-batch{c.batch_size:d}'
if opts.desc is not None:
    desc += f'-{opts.desc}'

##################################
########## StyleGAN-XL ###########
##################################

# Generator
c.G_kwargs.w_dim = 512
c.G_kwargs.z_dim = 64
c.G_kwargs.mapping_kwargs.rand_embedding = False
c.G_kwargs.num_layers = opts.syn_layers
c.G_kwargs.mapping_kwargs.num_layers = 2

# Discriminator
c.D_kwargs = dnnlib.EasyDict(
    class_name='pg_modules.discriminator.ProjectedDiscriminator',
    backbones=['deit_base_distilled_patch16_224', 'tf_efficientnet_lite0'],
    diffaug=True,
    interp224=(c.training_set_kwargs.resolution < 224),
    backbone_kwargs=dnnlib.EasyDict(),
)
c.D_kwargs.backbone_kwargs.cout = 64
c.D_kwargs.backbone_kwargs.expand = True
c.D_kwargs.backbone_kwargs.proj_type = 2 if c.training_set_kwargs.resolution <= 16 else 2  # CCM only works better on very low resolutions
c.D_kwargs.backbone_kwargs.num_discs = 4
c.D_kwargs.backbone_kwargs.cond = opts.cond

# Loss
c.loss_kwargs = dnnlib.EasyDict(class_name='training.loss.ProjectedGANLoss')
c.loss_kwargs.blur_init_sigma = 2  # Blur the images seen by the discriminator.
c.loss_kwargs.blur_fade_kimg = 300
c.loss_kwargs.pl_weight = 2.0
c.loss_kwargs.pl_no_weight_grad = True
c.loss_kwargs.style_mixing_prob = 0.0
c.loss_kwargs.cls_weight = 0.0  # use classifier guidance only for superresolution training (i.e., with pretrained stem)
c.loss_kwargs.cls_model = 'deit_small_distilled_patch16_224'
c.loss_kwargs.train_head_only = False

if opts.superres:
    assert opts.path_stem is not None, "When training superres head, provide path to stem"

    # Generator
    c.G_kwargs = dnnlib.EasyDict(
        class_name='training.networks_stylegan3_resetting.SuperresGenerator',
        path_stem=opts.path_stem,
        head_layers=opts.head_layers,
        up_factor=opts.up_factor,
        conv_kernel=1 if opts.cfg == 'stylegan3-r' else 3,
        use_radial_filters=True if opts.cfg == 'stylegan3-r' else False,
    )

    # Loss
    c.loss_kwargs.pl_weight = 0.0
    c.loss_kwargs.cls_weight = opts.cls_weight if opts.cond else 0
    c.loss_kwargs.train_head_only = True

##################################
##################################
##################################

print("Configuration set up successfully!")
print(f"Training will use {c.num_gpus} GPU(s)")
print(f"Batch size: {c.batch_size}, Batch per GPU: {c.batch_gpu}")
print(f"Dataset: {opts.data}")
print(f"Output directory: {opts.outdir}")

In [ ]:
# Launch training
launch_training(c=c, desc=desc, outdir=opts.outdir, dry_run=opts.dry_run)

# Check for restart
last_snapshot = misc.get_ckpt_path(c.run_dir)
if os.path.isfile(last_snapshot):
    # get current number of training images
    with dnnlib.util.open_url(last_snapshot) as f:
        cur_nimg = legacy.load_network_pkl(f)['progress']['cur_nimg'].item()
    if (cur_nimg//1000) < c.total_kimg:
        print('Restart: exit with code 3')
        exit(3)
